New development-region terrain

Apply the established notebook 02 terrain recipe to the new development sites
selected in the [EDA summary](../../EDA_PLAN.md). Default: Death Valley. Run Wadi Rum separately
after its archive is complete. This notebook never selects validation/final reserves.
Source receipts and actual footprints are checked before extraction. Native class
tiles are cropped and mosaicked before fraction aggregation, including south of
Death Valley's 36-degree boundary. Raw inputs and previous runs remain unchanged.
The 30 m grid is not a claim about effective source resolution. The natural
screen is exploratory and does not establish site access or analog relevance.

In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "outline.md").is_file() and (p / "data/raw").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the project root or a folder under notebooks")
sys.path.insert(0, str(ROOT / "notebooks"))
print("Project:", ROOT)

In [ ]:
import json
import math
import re
import shutil
import zipfile

import numpy as np
import rasterio
from processing import (
    Run,
    landcover_fractions,
    make_grid,
    mosaic_elevation,
    mosaic_landcover,
    pad_grid,
    roi_mask,
    select_covering_tiles,
    sha256,
    terrain,
    warp,
    write_raster,
)
from rasterio.crs import CRS
from rasterio.transform import array_bounds
from rasterio.warp import transform_bounds

AREAS = {
    "death_valley": {"bbox": (-117.10, 36.00, -116.60, 36.50), "crs": "EPSG:32611"},
    "wadi_rum": {"bbox": (35.20, 29.40, 35.65, 29.80), "crs": "EPSG:32636"},
}
SELECTED_AREAS = ["death_valley"]  # Only approved development additions; reserves are absent.
ARCHIVES = {
    "death_valley": ("N30W120-N40W110_FABDEM_V1-2.zip", "20261027T005824876538Z_run.json"),
    "wadi_rum": ("N20E030-N30E040_FABDEM_V1-2.zip", "20261029T074724109908Z_run.json"),
}
SPACING_M = 30
ROUGHNESS_WINDOW = 11  # 330 m at the default spacing; complete windows only.
HALO_CELLS = max(1, ROUGHNESS_WINDOW // 2)
INTERPOLATION_GUARD_CELLS = 2
MIN_BARE_FRACTION = 0.8  # Exploratory screening parameter; class 60 includes sparse vegetation.
config = {"areas": {name: AREAS[name] for name in SELECTED_AREAS}, "spacing_m": SPACING_M,
          "roughness_window_cells": ROUGHNESS_WINDOW, "minimum_bare_sparse_fraction": MIN_BARE_FRACTION,
          "elevation_resampling": "bilinear", "class_aggregation": "area-weighted binary indicators",
          "coverage_threshold": 0.99, "terrain_halo_cells": HALO_CELLS,
          "interpolation_guard_cells": INTERPOLATION_GUARD_CELLS,
          "tile_selection": "actual footprints cover padded processing grid plus interpolation guard",
          "relative_elevation_reference": "median elevation inside requested ROI",
          "role": "development", "landcover_mosaic": "native uint8 classes before area aggregation"}
assert SPACING_M >= 30 and 0 <= MIN_BARE_FRACTION <= 1

In [ ]:
archives = []
receipts = []
for name in SELECTED_AREAS:
    filename, receipt_name = ARCHIVES[name]
    archive = ROOT / "data/raw/earth/fabdem/archives" / filename
    receipt_path = ROOT / "data/raw/earth/fabdem/manifests" / receipt_name
    receipt = json.loads(receipt_path.read_text())
    assert receipt["status"] == "complete", "Complete acquisition before processing this region"
    record = next(item for item in receipt["completed"] if item["filename"] == filename)
    assert archive.stat().st_size == record["bytes"] and sha256(archive) == record["sha256"]
    archives.append(archive)
    receipts.append(receipt_path)
member_index = {}
for archive in archives:
    with zipfile.ZipFile(archive) as bundle:
        for item in bundle.infolist():
            if item.filename.lower().endswith(".tif"):
                name = Path(item.filename).name
                if name in member_index:
                    raise ValueError(f"Duplicate FABDEM member: {name}")
                member_index[name] = (archive, item.filename, item.CRC, item.file_size)
selected_tiles, cover_paths = {}, {}
output_grids, processing_grids = {}, {}
footprint_cache = {}
selection_bounds = {}
for name in SELECTED_AREAS:
    area = AREAS[name]
    projected = transform_bounds("EPSG:4326", area["crs"], *area["bbox"], densify_pts=41)
    grid = make_grid(projected, area["crs"], SPACING_M)
    work_grid = pad_grid(grid, HALO_CELLS)
    guard_grid = pad_grid(work_grid, INTERPOLATION_GUARD_CELLS)
    guard_bounds = array_bounds(guard_grid.height, guard_grid.width, guard_grid.transform)
    geographic = transform_bounds(area["crs"], "EPSG:4326", *guard_bounds, densify_pts=41)
    west, south, east, north = geographic
    candidates = {}
    # A one-degree search margin discovers neighboring names; final selection uses actual headers.
    for tile, (archive, member, _, _) in member_index.items():
        match = re.fullmatch(r"([NS])(\d{2})([EW])(\d{3})_FABDEM_V1-2.tif", tile)
        if match is None:
            continue
        lat = int(match[2]) * (1 if match[1] == "N" else -1)
        lon = int(match[4]) * (1 if match[3] == "E" else -1)
        if not (math.floor(south)-1 <= lat <= math.ceil(north) and math.floor(west)-1 <= lon <= math.ceil(east)):
            continue
        if tile not in footprint_cache:
            with rasterio.open(f"zip://{archive.resolve().as_posix()}!{member}") as src:
                if src.crs != CRS.from_epsg(4326):
                    raise ValueError(f"Unexpected FABDEM CRS: {tile}")
                footprint_cache[tile] = tuple(src.bounds)
        candidates[tile] = footprint_cache[tile]
    required = select_covering_tiles(candidates, geographic)
    selected_tiles[name] = required
    output_grids[name], processing_grids[name] = grid, work_grid
    selection_bounds[name] = geographic
    covers = {}
    for path in sorted((ROOT / "data/raw/earth/worldcover/v200/2021/map").glob("*.tif")):
        with rasterio.open(path) as src:
            assert src.crs == CRS.from_epsg(4326)
            covers[str(path)] = tuple(src.bounds)
    selected_covers = [Path(p) for p in select_covering_tiles(covers, geographic)]
    for path in selected_covers:
        receipt = json.loads(path.with_suffix(".tif.json").read_text())
        assert path.stat().st_size == receipt["size_bytes"] and sha256(path) == receipt["sha256"]
    cover_paths[name] = selected_covers
    print(name, "FABDEM:", required, "WorldCover:", [p.name for p in selected_covers])

config["selected_tiles"] = selected_tiles
config["source_selection_bounds_lonlat"] = selection_bounds
config["worldcover_tiles"] = {name: [p.name for p in paths] for name, paths in cover_paths.items()}

In [ ]:
with Run(ROOT, "earth_terrain_expansion", config, "preprocessing/19_development_region_terrain.ipynb") as run:
    import hashlib

    import nbformat
    source = "\n".join(c.source for c in nbformat.read(run.notebook, as_version=4).cells)
    run.record["notebook_source_sha256"] = hashlib.sha256(source.encode()).hexdigest()
    for receipt_path in receipts:
        run.input(receipt_path)
    run.input(ROOT / "EDA_PLAN.md")
    summaries = []
    for name in SELECTED_AREAS:
        area = AREAS[name]
        grid = output_grids[name]
        work_grid = processing_grids[name]
        core = (slice(HALO_CELLS, -HALO_CELLS), slice(HALO_CELLS, -HALO_CELLS))
        inside = roi_mask(grid, area["bbox"])
        extracted = []
        source_metadata = []
        for filename in selected_tiles[name]:
            archive, member, crc, size = member_index[filename]
            run.input(archive)
            target = run.path(f"earth/{name}/extracted/{filename}")
            with zipfile.ZipFile(archive) as bundle, bundle.open(member) as source, target.open("wb") as dst:
                shutil.copyfileobj(source, dst, 1024 * 1024)  # ZipExtFile checks member CRC at EOF.
            assert target.stat().st_size == size
            with rasterio.open(target) as src:
                source_metadata.append({"filename": filename, "crc32": crc, "crs": str(src.crs),
                                        "resolution": src.res, "nodata": src.nodata, "units": src.units,
                                        "bounds": list(src.bounds), "tags": src.tags()})
            extracted.append(target)
        mosaic = run.path(f"earth/{name}/native_mosaic.tif")
        mosaic_elevation(extracted, mosaic)
        elevation_with_halo = warp(mosaic, work_grid)
        work_inside = roi_mask(work_grid, area["bbox"])
        assert np.array_equal(work_inside[core], inside)
        features = terrain(elevation_with_halo, SPACING_M, ROUGHNESS_WINDOW, output_mask=work_inside)
        elevation = elevation_with_halo[core].copy()
        elevation[~inside] = np.nan
        del elevation_with_halo, work_inside
        folder = f"earth/{name}/grid_{SPACING_M}m"
        write_raster(run.path(f"{folder}/elevation_m.tif"), elevation, grid, "m", source="FABDEM V1-2")
        complete_features = np.isfinite(elevation)
        for feature, padded_values in features.items():
            values = padded_values[core]
            complete_features &= np.isfinite(values)
            unit = "degree" if feature == "slope_deg" else "1/m" if feature == "laplacian_per_m" else "m"
            write_raster(run.path(f"{folder}/{feature}.tif"), values, grid, unit,
                         spacing_m=SPACING_M, roughness_window=ROUGHNESS_WINDOW)
        del features
        for cover_path in cover_paths[name]:
            run.input(cover_path)
            run.input(cover_path.with_suffix(".tif.json"))
        cover = run.path(f"earth/{name}/worldcover_native_mosaic.tif")
        mosaic_landcover(cover_paths[name], cover, selection_bounds[name])
        fractions = landcover_fractions(cover, grid)
        for layer, values in fractions.items():
            values[~inside] = np.nan
            write_raster(run.path(f"{folder}/{layer}.tif"), values, grid, "fraction")
        valid = np.isfinite(elevation) & (fractions["source_coverage"] >= 0.99) & inside
        natural = valid & (fractions["built_fraction"] == 0) & (fractions["crop_fraction"] == 0)
        natural &= (fractions["water_fraction"] == 0) & (fractions["bare_sparse_fraction"] >= MIN_BARE_FRACTION)
        write_raster(run.path(f"{folder}/valid_input_mask.tif"), np.where(inside, valid, np.nan), grid, "boolean")
        write_raster(run.path(f"{folder}/natural_screen_mask.tif"), np.where(inside, natural, np.nan), grid, "boolean")
        summary = {"area": name, "grid_crs": str(grid.crs), "transform": list(grid.transform),
                   "shape": [grid.height, grid.width], "source_metadata": source_metadata,
                   "terrain_halo_cells": HALO_CELLS, "processing_shape": [work_grid.height, work_grid.width],
                   "joint_feature_fraction": float(complete_features[inside].mean()),
                   "screened_cells_missing_features": int((natural & ~complete_features).sum()),
                   "elevation_coverage": float(np.isfinite(elevation[inside]).mean()),
                   "valid_input_fraction": float(valid[inside].mean()), "natural_screen_fraction": float(natural[inside].mean())}
        run.path(f"earth/{name}/quality.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
        summaries.append(summary)
        print(name, "valid input fraction", summary["valid_input_fraction"])
        del elevation, fractions, valid, natural, inside, complete_features
    run.record["summaries"] = summaries
earth_output = run.directory
print("Completed:", earth_output)
